# When does blinking stop looking like the reference?

Every 30 seconds the app compares the last 5 minutes of blink intervals with your saved
**reference** (the first 10 minutes of one session). A window is **below the line** when its
p-value is under 0.01. A difference is **confirmed** only when **3 windows in a row** are below
the line. This notebook replays your recordings with the same code
(`ocusense/core/baseline_drift.py`) and shows **when** that happens and **which way** your
blinking changed. Recordings and the reference are only read, never changed. Set `RECORDINGS`
to show only some recordings.

In [ ]:
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ocusense").is_dir())
sys.path.insert(0, str(ROOT))

from ocusense import reference as reference_file
from ocusense.core import baseline_drift as bd
from ocusense.core.replay import replay
from ocusense.recording import default_directory
from ocusense.recording_files import RecordingError, group, read_file

FOLDER = default_directory()
RECORDINGS = None  # list of start times "YYYY-MM-DD HH:MM:SS" to show; None = all
ALPHA = bd.ALPHA  # 0.01, as in the app
STEP = bd.STEP_SECONDS  # 30 s
WINDOW_MIN = bd.WINDOW_SECONDS / 60  # 5 min
CONFIRM = bd.CONFIRM_WINDOWS  # 3 windows in a row below the line = confirmed different

# Colours (light theme): one blue for data, the "critical" red only for flagged results, greys for context.
BLUE = "#2a78d6"
BLUE_LIGHT = "#b7d3f6"
RED = "#d03b3b"
RED_LIGHT = "#efb1b1"
INK = "#0b0b0b"
INK_2 = "#52514e"
GRID = "#e4e3de"
UNTESTED = "#d9d8d2"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": GRID, "axes.labelcolor": INK_2,
    "xtick.color": INK_2, "ytick.color": INK_2, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10, "axes.titlesize": 12,
    "axes.titleweight": "bold", "axes.titlelocation": "left", "legend.frameon": False,
})

## 1. Load the reference and replay every recording

Each recording is replayed with its own calibration and split into 5-minute windows every 30 s.
Each window is tested against the reference. If a recording *is* the reference, its first 10
minutes overlap it.

In [ ]:
ref = reference_file.load()
if ref is None:
    raise SystemExit("No saved reference yet: run analysis/set_reference.py, or let the app save one.")
ref_low, ref_high = math.exp(ref.mu - 0.6745 * ref.sigma), math.exp(ref.mu + 0.6745 * ref.sigma)
print(f"Reference: session of {ref.label()} · n = {ref.n} · median {ref.median:.2f} s · "
      f"middle 50 % {ref_low:.1f}–{ref_high:.1f} s")

files = []
for path in sorted(FOLDER.glob("ear_*.csv")):
    try:
        f = read_file(path)
    except RecordingError:
        continue
    if f.calibration is not None:
        files.append(f)


def started(r):
    return f"{r.started:%Y-%m-%d %H:%M:%S}"


def is_reference(r):
    return r.started.isoformat(timespec="seconds") == ref.started_at


chosen = [r for r in group(files) if RECORDINGS is None or started(r) in RECORDINGS]
analyses = []
for r in chosen:
    result = replay(r.rows, r.calibration)
    own = is_reference(r)
    windows = bd.sliding_windows(result.distributions.intervals, 0.0, result.timeline.times[-1],
                                 baseline=ref.intervals, baseline_seconds=ref.seconds if own else 0.0)
    analyses.append({"recording": r, "result": result, "windows": windows, "own": own,
                     "reminders": [t / 60 for t in bd.break_reminders(windows, ref.mu)],  # when the app would remind
                     "name": f"{r.started:%a %d %b, %H:%M}" + (" (reference)" if own else ""),
                     "minutes": result.timeline.times[-1] / 60})
print(f"{len(analyses)} recording(s)")


def state(w):
    if w.p is None:
        return "untested"
    if w.confirmed:
        return "confirmed"
    return "below" if w.different else "similar"


def shown_at(w):
    """When the app would show this result: the window's end, in minutes."""
    return w.end / 60


def centre(w):
    """The middle of the 5 minutes the result describes, in minutes."""
    return (w.start + w.end) / 2 / 60


def describe(run):
    """How a flagged stretch differs from the reference: level (median) and spread (sigma)."""
    mu = float(np.mean([w.fit.mu for w in run]))
    sigma = float(np.mean([w.fit.sigma for w in run]))
    if mu - ref.mu > 0.15:
        level = "blinking less often (longer intervals)"
    elif ref.mu - mu > 0.15:
        level = "blinking more often (shorter intervals)"
    else:
        level = "about the reference's median"
    if sigma > 1.25 * ref.sigma:
        level += ", more irregular (wider spread)"
    elif sigma < 0.8 * ref.sigma:
        level += ", more regular (narrower spread)"
    return level


def runs_below(windows):
    """Runs of consecutive windows below the line (each a list of windows)."""
    runs, current = [], []
    for w in windows:
        if w.different:
            current.append(w)
        elif current:
            runs.append(current)
            current = []
    if current:
        runs.append(current)
    return runs


def episodes(windows):
    """Confirmed stretches: runs of at least CONFIRM windows below the line."""
    return [run for run in runs_below(windows) if len(run) >= CONFIRM]

## 2. Overview: what the app would have shown, minute by minute

One row per recording. Each window's result is drawn from the moment the app would show it (the
end of its 5 minutes) until the next result 30 s later.
- **Full red = confirmed different**: this window and the two before it were all below the line.
- **Light red** = below the line but not (yet) confirmed: a short dip.
- **Hatched** = too few blinks to test (usually poor tracking or a pause). It breaks a run.
- The first 5 minutes have no result yet.
- **▼ = break reminder**: 3 windows in a row below the line with longer intervals than the
  reference (blinking less often). At most one per slow stretch, at least 20 min apart, exactly
  as the app does it. Faster blinking never triggers one.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 0.7 * len(analyses) + 1.4))
colours = {"similar": BLUE_LIGHT, "below": RED_LIGHT, "confirmed": RED, "untested": UNTESTED}
for row, a in enumerate(analyses):
    y = len(analyses) - 1 - row
    ax.broken_barh([(0, a["minutes"])], (y - 0.3, 0.6), facecolors="none", edgecolors=GRID, linewidth=1)
    for w in a["windows"]:
        # each result holds until the next one, STEP later (the last one until the recording ends)
        start = shown_at(w)
        width = min(STEP / 60, a["minutes"] - start) if w is not a["windows"][-1] else max(a["minutes"] - start, 0.05)
        ax.broken_barh([(start, width)], (y - 0.3, 0.6), facecolors=colours[state(w)], edgecolors="none",
                       hatch="////" if state(w) == "untested" else None)
    ax.scatter(a["reminders"], [y + 0.45] * len(a["reminders"]), marker="v", s=70, color=INK, zorder=4, clip_on=False)
    stretches = len(episodes(a["windows"]))
    dips = len(runs_below(a["windows"])) - stretches
    ax.text(a["minutes"] + 0.8, y, f"{stretches} confirmed stretch{'es' if stretches != 1 else ''} · "
            f"{dips} short dip{'s' if dips != 1 else ''} · {len(a['reminders'])} break reminder"
            f"{'s' if len(a['reminders']) != 1 else ''}", va="center", color=INK_2, fontsize=9)
ax.set_yticks(range(len(analyses)), [a["name"] for a in reversed(analyses)], color=INK)
ax.set_xlabel("minutes since the recording started (each result shown from the end of its 5-minute window)")
ax.set_xlim(0, max(a["minutes"] for a in analyses) * 1.32)
ax.set_ylim(-0.7, len(analyses) - 0.1)
ax.grid(axis="y", visible=False)
ax.set_title("When does blinking stop looking like the reference?")
fig.legend(handles=[Patch(facecolor=BLUE_LIGHT, label="similar to the reference"),
                   Patch(facecolor=RED_LIGHT, label=f"below p = {ALPHA:g}, not confirmed"),
                   Patch(facecolor=RED, label=f"confirmed different ({CONFIRM} in a row)"),
                   Patch(facecolor=UNTESTED, hatch="////", label="not enough data (< 15 intervals)"),
                   Patch(facecolor="none", edgecolor=GRID, label="first 5 min: no result yet"),
                   Line2D([], [], marker="v", linestyle="none", color=INK, markersize=8,
                          label="break reminder (slower blinking, confirmed)")],
           loc="lower center", ncol=3, fontsize=9)
plt.tight_layout(rect=(0, 0.16, 1, 1))
plt.show()

## 3. Each recording in detail

- **Top:** the p-value of each window (log scale) and the dashed 0.01 line. **Filled red dots** are
  confirmed windows; **hollow red dots** are windows below the line that weren't confirmed. Grey
  ticks at the bottom are windows with too few intervals.
- **Bottom:** what's underneath. Grey dots are every single interval between two blinks. The blue
  line is the median of each 5-minute window. The grey band is the reference's middle 50 %, with
  its median dashed. Above the band = blinking less often than in the reference; below = more often.
- Results are plotted at the **middle** of the 5 minutes they describe, so they line up with the
  dots. The overview above uses the window's end instead (when the app would show it). Confirmed
  stretches are shaded from the window that confirmed them.

In [ ]:
def plot_recording(a):
    windows = a["windows"]
    tested = [w for w in windows if w.p is not None]
    flagged = [w for w in windows if w.confirmed]
    dips = [w for w in windows if w.different and not w.confirmed]
    pairs = a["result"].distributions.intervals
    fig, (top, bottom) = plt.subplots(2, 1, figsize=(12, 6.4), sharex=True, gridspec_kw={"height_ratios": [1, 1.2]})
    fig.suptitle(f"{a['recording'].label()}" + (" — the reference itself" if a["own"] else ""),
                 x=0.01, ha="left", fontsize=13, fontweight="bold", color=INK)

    for ax in (top, bottom):
        for run in episodes(windows):
            first = next(w for w in run if w.confirmed)  # shaded from the window that confirmed it
            ax.axvspan(centre(first) - STEP / 120, centre(run[-1]) + STEP / 120, color=RED, alpha=0.08, linewidth=0)
        if a["own"]:
            ax.axvspan(0, ref.seconds / 60, color=INK_2, alpha=0.06, linewidth=0)

    # p-value: lower = less like the reference; below the dashed line = flagged
    p = [w.p if w.p is not None else np.nan for w in windows]
    top.plot([centre(w) for w in windows], p, color=BLUE, linewidth=2)
    top.scatter([centre(w) for w in flagged], [w.p for w in flagged], s=26, color=RED, zorder=3,
                edgecolors="#fcfcfb", linewidths=1)
    top.scatter([centre(w) for w in dips], [w.p for w in dips], s=26, facecolors="#fcfcfb", edgecolors=RED,
                linewidths=1.5, zorder=3)
    top.axhline(ALPHA, color=RED, linestyle="--", linewidth=1)
    top.set_yscale("log")
    low = min([w.p for w in tested] + [ALPHA / 10])
    top.set_ylim(low / 3, 1.5)
    untested = [w for w in windows if w.p is None]
    top.scatter([centre(w) for w in untested], [low / 2] * len(untested), marker="|", s=60, color=INK_2)
    top.set_ylabel("p-value (log scale)")
    top.set_title("How likely is this 5-minute window under the reference?", fontsize=11)

    # the data underneath: every interval, and each window's median against the reference's middle 50 %
    onsets = np.array([b for _a, b in pairs]) / 60
    values = np.array([b - a_ for a_, b in pairs])
    bottom.scatter(onsets, values, s=6, color=INK_2, alpha=0.25, linewidths=0)
    bottom.axhspan(ref_low, ref_high, color=INK_2, alpha=0.10, linewidth=0)
    bottom.axhline(ref.median, color=INK_2, linestyle="--", linewidth=1)
    med = [math.exp(w.fit.mu) if w.p is not None else np.nan for w in windows]
    bottom.plot([centre(w) for w in windows], med, color=BLUE, linewidth=2)
    bottom.scatter([centre(w) for w in flagged], [math.exp(w.fit.mu) for w in flagged], s=26, color=RED, zorder=3,
                   edgecolors="#fcfcfb", linewidths=1)
    bottom.scatter([centre(w) for w in dips], [math.exp(w.fit.mu) for w in dips], s=26, facecolors="#fcfcfb",
                   edgecolors=RED, linewidths=1.5, zorder=3)
    bottom.set_yscale("log")
    bottom.set_ylim(0.25, 40)
    bottom.set_yticks([0.5, 1, 2, 5, 10, 20], ["0.5", "1", "2", "5", "10", "20"])
    bottom.set_ylabel("time between blinks (s, log scale)")
    bottom.set_xlabel("minutes since the recording started (each window's result at the middle of its 5 minutes)")
    bottom.set_title("Above the band: blinking less often than the reference · below: more often", fontsize=11)
    bottom.legend(handles=[
        Line2D([], [], color=BLUE, linewidth=2, label="median of each 5-minute window"),
        Line2D([], [], color=RED, linestyle="--", linewidth=1, label=f"p = {ALPHA:g} line (top)"),
        Line2D([], [], marker="o", linestyle="none", color=RED, label=f"confirmed ({CONFIRM}+ in a row below p = {ALPHA:g})"),
        Line2D([], [], marker="o", linestyle="none", markerfacecolor="#fcfcfb", markeredgecolor=RED,
               markeredgewidth=1.5, label=f"below p = {ALPHA:g}, not confirmed"),
        Patch(facecolor=INK_2, alpha=0.18, label=f"reference middle 50 % ({ref_low:.1f}–{ref_high:.1f} s)"),
        Line2D([], [], color=INK_2, linestyle="--", label=f"reference median ({ref.median:.1f} s)"),
        Line2D([], [], marker="o", linestyle="none", color=INK_2, alpha=0.4, markersize=4, label="each interval"),
        Patch(facecolor=RED, alpha=0.15, label="confirmed stretch"),
        Line2D([], [], color=INK, linestyle=":", linewidth=1.2, label="break reminder"),
    ], loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3, fontsize=9)
    fig.subplots_adjust(bottom=0.24)
    for t in a["reminders"]:  # at the middle of the window that triggered it (it ends at t)
        for ax in (top, bottom):
            ax.axvline(t - WINDOW_MIN / 2, color=INK, linewidth=1.2, linestyle=":")
        bottom.annotate("break reminder", xy=(t - WINDOW_MIN / 2, 0.0), xycoords=("data", "axes fraction"),
                        xytext=(4, 4), textcoords="offset points", fontsize=9, color=INK,
                        bbox=dict(facecolor="#fcfcfb", edgecolor="none", pad=1))
    top.set_xlim(0, a["minutes"] + 0.5)
    plt.tight_layout()
    plt.show()


for a in analyses:
    plot_recording(a)

## 4. Confirmed stretches

Each run of 3 or more windows in a row below the line, merged into one stretch. For each: when
it was confirmed and how long it lasted (as the app would have shown it), which minutes of data
it covers, the window medians against the reference, and how it differs. "Level" compares the
median (mu); "spread" compares sigma, meaning more or less regular intervals. Shorter dips are
only counted.

In [ ]:
for a in analyses:
    print(f"\n{a['name']}: break reminders at " + (", ".join(f"{t:.1f} min" for t in a["reminders"]) or "—"))
    runs = episodes(a["windows"])
    dips = len(runs_below(a["windows"])) - len(runs)
    print(f"\n{a['name']}: {len(runs)} confirmed stretch(es); {dips} short dip(s) below the line, not confirmed")
    for run in runs:
        confirmed_at = shown_at(next(w for w in run if w.confirmed))
        medians = [math.exp(w.fit.mu) for w in run]
        print(f"  confirmed at {confirmed_at:5.1f} min, lasted until {shown_at(run[-1]) + STEP / 60:5.1f} min · "
              f"{len(run):2d} windows below the line, "
              f"data {run[0].start / 60:.1f}–{run[-1].end / 60:.1f} min · median {min(medians):.1f}–{max(medians):.1f} s "
              f"(reference {ref.median:.1f} s) · {describe(run)} · lowest p {min(w.p for w in run):.1g}")

## Reading these plots

- About 100 windows are tested per hour and neighbours share 90 % of their data, so single windows
  below the line happen by chance. That's why only 3 in a row count as different.
- "Different" means *not distributed like the reference's 10 minutes*. Another activity (long
  reading, talking, skimming) counts just as much as drift or tiredness. It is not a fatigue score.
- Recordings made with a **different calibration** than the reference (or with poor tracking) are
  less comparable: other thresholds can count blinks differently, and gaps drop long intervals.